# Load and analyze funds (V2)


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 1. Load the data

Load the **Funds** sheet of `CLEAN_RECOLTE DONNEES-EXCEL-V2.xlsx` from `input-data-alix/clean/`.
Then place the selected funds in a new dataframe called `selected_funds`

</div>

In [36]:
import sys
from pathlib import Path


def _ensure_helpers_importable() -> None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


_ensure_helpers_importable()

from helpers.funds_functions.load_workbook import find_workbook, load_workbook

FILENAME = "CLEAN_RECOLTE DONNEES-EXCEL-V2.xlsx"
workbook = find_workbook(Path("input-data-alix") / "clean" / FILENAME)
funds = load_workbook(workbook)

print(f"Loaded {workbook}\n")
print(f"Funds: {len(funds)} rows, {len(funds.columns)} columns")

selected_funds = funds.loc[funds["fund_type"].notna() & funds["fund_type"].astype("string").str.strip().ne("")].copy()

print(f"selected_funds: {len(selected_funds)} rows")
selected_funds["fund_type"].value_counts()


Loaded /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/input-data-alix/clean/CLEAN_RECOLTE DONNEES-EXCEL-V2.xlsx

Funds: 2610 rows, 65 columns
selected_funds: 606 rows


fund_type
Active      514
Passive      72
To Check     20
Name: count, dtype: int64

<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 2. Verify if ISIN and CUSIP are unique

</div>

In [37]:
from IPython.display import display


def _filled_identifier(series):
    values = series.astype("string").str.strip()
    blank = values.isna() | values.eq("") | values.str.lower().eq("<na>")
    return values.mask(blank)


def _identifier_label(isin_flag, cusip_flag):
    parts = []
    if isin_flag:
        parts.append("ISIN")
    if cusip_flag:
        parts.append("CUSIP")
    return " & ".join(parts)


isin = _filled_identifier(selected_funds["ISIN"])
cusip = _filled_identifier(selected_funds["CUSIP"])

for column, values in (("ISIN", isin), ("CUSIP", cusip)):
    filled = values.dropna()
    repeated = filled[filled.duplicated(keep=False)]
    status = "unique" if repeated.empty else f"{repeated.nunique()} repeated values"
    print(
        f"{column}: {filled.nunique()} unique values "
        f"among {len(filled)} non-empty ({values.isna().sum()} missing) — {status}"
    )

isin_shared = isin.notna() & isin.duplicated(keep=False)
cusip_shared = cusip.notna() & cusip.duplicated(keep=False)
shared_funds = selected_funds.loc[isin_shared | cusip_shared, ["Name", "ISIN", "CUSIP", "fund_type"]].copy()
if shared_funds.empty:
    print("\nNo funds share an ISIN or a CUSIP")
else:
    shared_funds.insert(
        0,
        "shared_identifier",
        [
            _identifier_label(isin_flag, cusip_flag)
            for isin_flag, cusip_flag in zip(
                isin_shared.loc[shared_funds.index],
                cusip_shared.loc[shared_funds.index],
            )
        ],
    )
    shared_funds = shared_funds.assign(_sort=isin.where(isin_shared, cusip)).sort_values(
        ["_sort", "Name"]
    ).drop(columns="_sort")
    print(f"\n{len(shared_funds)} funds share an ISIN or a CUSIP")
    display(shared_funds)

missing_isin = isin.isna()
missing_cusip = cusip.isna()
missing_funds = selected_funds.loc[missing_isin | missing_cusip, ["Name", "ISIN", "CUSIP", "fund_type"]].copy()
if missing_funds.empty:
    print("\nNo funds are missing an ISIN or a CUSIP")
else:
    missing_funds.insert(
        0,
        "missing_identifier",
        [
            _identifier_label(isin_flag, cusip_flag)
            for isin_flag, cusip_flag in zip(
                missing_isin.loc[missing_funds.index],
                missing_cusip.loc[missing_funds.index],
            )
        ],
    )
    missing_funds = missing_funds.sort_values(["missing_identifier", "Name"])
    print(f"\n{len(missing_funds)} funds are missing an ISIN or a CUSIP")
    display(missing_funds)


ISIN: 606 unique values among 606 non-empty (0 missing) — unique
CUSIP: 605 unique values among 605 non-empty (1 missing) — unique

No funds share an ISIN or a CUSIP

1 funds are missing an ISIN or a CUSIP


,missing_identifier,Name,ISIN,CUSIP,fund_type
14,CUSIP,American Funds Growth Fund of America;R1,US3998748749,NaN,Active


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

**&#9888;&#65039; Dataset enrichment**

A US ISIN is `US`, then the 9-character CUSIP, then one check digit. For each selected fund with an empty CUSIP and a US ISIN, that middle value is sent to OpenFIGI (`ID_CUSIP`).

The CUSIP is written into `selected_funds` only when OpenFIGI returns the same listing as the ISIN. The same FIGI means the same listing. Any other result is left blank.

</div>

In [38]:
from IPython.display import display

from helpers.funds_functions.check_openfigi_isin import record_from_job, same_openfigi_record
from helpers.funds_functions.map_market_identifiers import cusip_shape_ok, post_mapping


def _blank_identifier(series):
    values = series.astype("string").str.strip()
    return values.isna() | values.eq("") | values.str.lower().eq("<na>")


def _cusip_from_us_isin(isin):
    if not isinstance(isin, str) or len(isin) != 12 or not isin.startswith("US"):
        return None
    cusip = isin[2:11]
    return cusip if cusip_shape_ok(cusip) else None


isin_text = selected_funds["ISIN"].astype("string").str.strip().str.upper()
candidate = isin_text.map(_cusip_from_us_isin)
missing_cusip = _blank_identifier(selected_funds["CUSIP"])
targets = selected_funds.loc[missing_cusip & candidate.notna(), ["Name", "ISIN"]].copy()
targets["CUSIP"] = candidate.loc[targets.index]

if targets.empty:
    print("No missing CUSIP can be read from a US ISIN")
else:
    isins = list(dict.fromkeys(targets["ISIN"].astype(str)))
    cusips = list(dict.fromkeys(targets["CUSIP"].astype(str)))
    isin_records = {
        isin: record_from_job(job)
        for isin, job in zip(isins, post_mapping(isins, "ID_ISIN"))
    }
    cusip_records = {
        cusip: record_from_job(job)
        for cusip, job in zip(cusips, post_mapping(cusips, "ID_CUSIP"))
    }
    confirmed = [
        same_openfigi_record(isin_records.get(isin), cusip_records.get(cusip))
        for isin, cusip in zip(targets["ISIN"].astype(str), targets["CUSIP"].astype(str))
    ]
    targets["openfigi_ticker"] = [
        (cusip_records.get(cusip) or {}).get("ticker") if ok else None
        for cusip, ok in zip(targets["CUSIP"].astype(str), confirmed)
    ]
    targets["openfigi_name"] = [
        (cusip_records.get(cusip) or {}).get("name") if ok else None
        for cusip, ok in zip(targets["CUSIP"].astype(str), confirmed)
    ]
    targets["filled"] = confirmed
    accepted = targets.index[targets["filled"]]
    selected_funds.loc[accepted, "CUSIP"] = targets.loc[accepted, "CUSIP"].astype("string")
    print(f"Filled {len(accepted)} of {len(targets)} missing CUSIPs")
    display(
        targets.loc[:, ["Name", "ISIN", "CUSIP", "openfigi_ticker", "openfigi_name", "filled"]]
    )

Filled 1 of 1 missing CUSIPs


,Name,ISIN,CUSIP,openfigi_ticker,openfigi_name,filled
14,American Funds Growth Fund of America;R1,US3998748749,399874874,RGAAX,AMER FND GRW FD OF AM-R1,True


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 4. Verify ISINs and enrich dataset with OpenFIGI

Each ISIN is sent to the OpenFIGI mapping API (`ID_ISIN`).

- `isin_found` is true when OpenFIGI returns a record.
- `name_matches` compares the sheet name with that record's name. OpenFIGI short forms (`LRG`, `INST`, `AMER FND`) count as the same name. A different word, such as Core versus Growth, does not.
- Each file CUSIP is also sent to OpenFIGI (`ID_CUSIP`). `cusip_matches` is true when that lookup returns the same listing as the ISIN. The same FIGI means the same listing. A missing CUSIP, a miss, or a different listing is false.
- The ISIN record's ticker and name are written to `openfigi_market_id` and `openfigi_name`. A different value already in either column is replaced, and the cell prints an alert.

ISIN records are cached in `cache/cache-isin-openfigi-record.json`. CUSIP listings are cached in `cache/cache-cusip-openfigi.json`.

The list of funds that are not in OpenFIGI are placed in a dataframe with the funds that don't match in OpenFigi : `unmatched_funds`

</div>

In [39]:
from helpers.funds_functions.check_openfigi_isin import check_openfigi_isins

selected_funds = check_openfigi_isins(selected_funds)
selected_funds.loc[
    :,
    ["Name", "openfigi_name", "ISIN", "isin_found", "name_matches", "CUSIP", "cusip_matches"],
].head()


599 of 606 ISINs found in OpenFIGI
324 of 606 names match the OpenFIGI record
599 of 606 CUSIPs return the same OpenFIGI record as the ISIN


,Name,openfigi_name,ISIN,isin_found,name_matches,CUSIP,cusip_matches
0,Auxier Focus Fund;Investor,AUXIER FOCUS FUND-INV,US3499031793,True,True,349903179,True
14,American Funds Growth Fund of America;R1,AMER FND GRW FD OF AM-R1,US3998748749,True,True,399874874,True
20,Principal LargeCap S&P 500 Index Fund;Inst,PRINCIPAL L/C S&P 500-IS,US74253Q7887,True,True,74253Q788,True
27,Fidelity Advisor Large Cap Stock Fund;M,FIDELITY ADV LRG CAP STK-M,US3159114202,True,False,315911420,True
34,DFA US Large Company Portfolio;Inst,DFA US LRG COMP PORT-INST,US25434D8231,True,True,25434D823,True


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

**&#9888;&#65039; Some records have no OpenFIGI match an will have no market ticker and no possibility to match with EDGAR**

Funds with no match are saved as `unmatched_funds` in `output-data/v2-no-matching-OPENFIGI.xlsx`.

</div>

In [40]:
unmatched_funds = selected_funds.loc[~selected_funds["isin_found"]].copy()


def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


openfigi_export = _output_dir() / "v2-no-matching-OPENFIGI.xlsx"
unmatched_funds.to_excel(openfigi_export, index=False)
print(f"Wrote {len(unmatched_funds)} rows to {openfigi_export}")


Wrote 7 rows to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/v2-no-matching-OPENFIGI.xlsx


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 5. Verify and enrich ISINs with EDGAR

The SEC fund reference is keyed by ticker. Each ISIN is resolved with its OpenFIGI ticker, then looked up in that reference. The CUSIP resolves to the same ticker, so it is not looked up separately.

- `edgar_isin_found` is true when the ISIN's ticker has an EDGAR record.
- `edgar_name` is that row's series name and share class. It is empty when the ISIN does not match.
- `edgar_name_match` compares the sheet name with `edgar_name`. Short forms count as the same name, and a different word such as Core versus Growth does not. It is false when `edgar_name` is empty.
- `edgartools_cik` is the registrant. One CIK is the investment company.
- `edgar_company_name` is that registrant's name.
- `edgartools_series_id` is the series inside the registrant.
- `edgar_series_name` is that series name.
- `edgartools_class_id` is the share class inside the series.
- `edgar_class_name` is that share class name.

The CIK, series, and share class fields come from the ISIN's EDGAR row. They are empty when the ISIN does not match.

</div>

In [44]:
from helpers.funds_functions.match_edgar_identifiers import match_edgar_identifiers

selected_funds = match_edgar_identifiers(selected_funds)
# selected_funds.loc[
#     :,
#     [
#         "Name",
#         "edgar_name",
#         "openfigi_name",
#         "edgar_name_match",
#         "ISIN",
#         "edgar_isin_found",
#         "CUSIP",
#     ],
# ].head()



43,121 EDGAR fund reference rows loaded from the cache
563 of 606 ISINs match an EDGAR record
563 of 606 CUSIPs match an EDGAR record
481 of 563 EDGAR names match the sheet


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

**&#9888;&#65039; Some records have no EDGAR match an will not be enriched**

Funds with no match are saved as `unmatched_funds` in `output-data/v2-no-matching-EDGAR.xlsx`.

</div>

In [45]:
unmatched_funds = selected_funds.loc[~selected_funds["edgar_isin_found"]].copy()

print(f"{len(unmatched_funds)} funds have no EDGAR record")
# unmatched_funds.loc[
#     :,
#     [
#         "Name",
#         "ISIN",
#         "edgar_isin_found",
#         "CUSIP",
#         "openfigi_market_id",
#         "fund_type",
#         "fund_wrapper",
#     ],
# ]
def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


edgar_export = _output_dir() / "v2-no-matching-EDGAR.xlsx"
unmatched_funds.to_excel(edgar_export, index=False)
print(f"\nWrote {len(unmatched_funds)} rows to {edgar_export}")



43 funds have no EDGAR record

Wrote 43 rows to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/v2-no-matching-EDGAR.xlsx


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 6. Verify if some funds belong to the same series

Funds are filed in EDGAR as a registrant (CIK), then a series, then a share class. One CIK is the investment company. It can offer several series, and each series can offer several share classes of the same portfolio.

The selected funds are all supposed to be primary share classes : the main share class among many. Therefor we don't expect to see rows in our table that belong to the sames cik/series.

</div>

In [49]:
from IPython.display import display

class_id = selected_funds["edgartools_class_id"].astype("string").str.strip()
has_class_id = class_id.notna() & class_id.ne("") & class_id.str.lower().ne("<na>")
missing_class_id = selected_funds.loc[
    ~has_class_id,
    ["Name", "ISIN", "openfigi_market_id", "edgartools_cik", "edgartools_series_id"],
]

cik = selected_funds["edgartools_cik"].astype("string").str.strip()
series_id = selected_funds["edgartools_series_id"].astype("string").str.strip()
has_pair = (
    cik.notna()
    & cik.ne("")
    & cik.str.lower().ne("<na>")
    & series_id.notna()
    & series_id.ne("")
    & series_id.str.lower().ne("<na>")
)
pair = cik + " ; " + series_id
pair_counts = pair.loc[has_pair].value_counts()
shared_pairs = pair_counts.loc[pair_counts.gt(1)]
same_cik_series = selected_funds.loc[pair.isin(shared_pairs.index)].sort_values(
    ["edgartools_cik", "edgartools_series_id", "Name"]
)


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

**&#9888;&#65039; Some records have no EDGAR share class id**

Funds with no match are saved as `unmatched_funds` in `output-data/v2-no-matching-EDGAR_SHARE_CLASS.xlsx`.

</div>

In [ ]:
def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


share_class_export = _output_dir() / "v2-no-matching-EDGAR_SHARE_CLASS.xlsx"
missing_class_id.to_excel(share_class_export, index=False)
print(f"Wrote {len(missing_class_id)} rows to {share_class_export}")


Wrote 43 rows to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/v2-no-matching-EDGAR_SHARE_CLASS.xlsx


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

**&#9888;&#65039; Some records share classes belonging to the same fund series**

Funds with no match are saved as `unmatched_funds` in `output-data/v2-common-fund-series.xlsx`.

</div>

In [ ]:
def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


common_series = same_cik_series.loc[
    :,
    [
        "edgartools_cik",
        "edgar_company_name",
        "edgartools_series_id",
        "edgar_series_name",
        "edgartools_class_id",
        "edgar_class_name",
        "Name",
        "ISIN",
    ],
]
common_series_export = _output_dir() / "v2-common-fund-series.xlsx"
common_series.to_excel(common_series_export, index=False)
print(f"Wrote {len(common_series)} rows to {common_series_export}")


Wrote 383 rows to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/v2-common-fund-series.xlsx


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 7. Verify fund type (active vs passive)

Each fund is classified at the EDGAR series: one registrant (CIK) and one series id. Share classes of that series receive the same label. The label is Passive, Active, Other, or Unknown. The first source that decides wins.

1. The latest N-CEN filing for the CIK. The questionnaire for `edgartools_series_id` is read from the filing XML.
   - `indexFundInfo` with fund type Index Fund means Passive. That block is the SEC census flag that the series tracks an index.
   - A fund type of Target Date Fund, Fund of Funds, or Interval Fund, with no `indexFundInfo`, means Other.
   - Another fund type, including Exchange-Traded Fund, with no `indexFundInfo`, means Active. An ETF is a wrapper. The SEC adds `indexFundInfo` when that series is an index fund.
   - A fund type of N/A, or a series id absent from that filing, moves to the next source.
2. When N-CEN does not decide, Jev (`jev-latest`) reads the prospectus and chooses Passive, Active, or Other.
   - The text is the whole 497K investment objective for that series.
   - When the 497K has no objective, the text is the investment-objective section and the principal-strategy section of the latest 485BPOS or 485APOS. The table of contents is left out.
   - Passive means the fund's job is to track, match, or approximate an index.
   - Active means a manager selects securities. An index named only as a comparison, or as a list of stocks the manager may buy, does not make the fund passive.
   - Other means a target-date fund, a fund of funds, or an interval fund.
   - A choice with confidence below 0.60 stays Unknown.
3. A row with no CIK or no series id is Unknown and is not sent to EDGAR. A series with no prospectus text is Unknown and is not sent to Jev.

The parsed N-CEN field `fund_type` is left unused: edgartools copies a single `fundType` tag and drops `indexFundInfo`. Portfolio turnover and the word Index in the fund name are left unused.
</div>

In [56]:
import importlib

import helpers.funds_functions.classify_fund_management_style as style_mod
import helpers.funds_functions.fund_management_style_rules as rules_mod

importlib.reload(rules_mod)
importlib.reload(style_mod)

selected_funds = style_mod.classify_fund_management_style(selected_funds)


286 EDGAR series to classify
N-CEN failed for CIK nan: CIK must be string or integer, got <class 'float'>
497K failed for nan: CIK must be string or integer, got <class 'float'>
485 failed for nan: CIK must be string or integer, got <class 'float'>
14 of 286 nan: Unknown (none)
380 Active, 150 Unknown, 53 Passive, 23 Other


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

**&#9888;&#65039; Some funds have an invalidated fund type**

Funds whose sheet fund type does not match the EDGAR fund type are saved as `fund_type_mismatch` in `output-data/v2-fund-type-mismatch.xlsx`.

</div>

In [59]:
def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


def _type_label(series):
    values = series.astype("string").str.strip()
    blank = values.isna() | values.eq("") | values.str.lower().eq("<na>")
    return values.mask(blank)


sheet_type = _type_label(selected_funds["fund_type"])
edgar_type = _type_label(selected_funds["fund_management_style"])
fund_type_mismatch = selected_funds.loc[
    sheet_type.fillna("").str.casefold().ne(edgar_type.fillna("").str.casefold())
].copy()

fund_type_export = _output_dir() / "v2-fund-type-mismatch.xlsx"
fund_type_mismatch.to_excel(fund_type_export, index=False)
print(
    f"{len(fund_type_mismatch)} funds have a fund type that does not match the EDGAR fund type"
)
print(f"Wrote {len(fund_type_mismatch)} rows to {fund_type_export}")


185 funds have a fund type that does not match the EDGAR fund type
Wrote 185 rows to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/v2-fund-type-mismatch.xlsx


<div style="box-sizing:border-box; max-width:calc(100% - 32px); background-color:rgba(59,130,246,0.12); padding:12px; border-radius:6px; border-left:4px solid #3b82f6">

### 8. Verify if there is a time series

Load the **D-Timeseries** sheet of the same workbook. The first rows are labels, not a column header. A daily total return series is a column whose note is `Total return index, daily` and that has at least one value.

Each selected fund is matched on ISIN. The Datastream name on that column is then checked against the fund. The sheet name is used first. When that name does not confirm the series, the OpenFIGI name is used, then the EDGAR name. A shortened Datastream name still counts as the same fund. A different word, such as Value versus Equity, does not.

`has_daily_total_return` is true when the series exists. `daily_tr_observations` is the number of daily points. `daily_tr_name` is the Datastream name. `daily_tr_name_match` is true when one of the three names confirms it, and `daily_tr_name_source` is the name that did.

Funds with no series are written to `output-data/v2-no-daily-total-return.xlsx`. Funds whose series name does not match are written to `output-data/v2-no-matching-D-TIMESERIES-NAME.xlsx`.

</div>

In [60]:
import pandas as pd

from helpers.funds_functions.load_workbook import find_workbook
from helpers.funds_functions.match_daily_total_return import match_daily_total_return


def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


daily_timeseries = pd.read_excel(
    find_workbook(Path("input-data-alix") / "clean" / FILENAME),
    sheet_name="D-Timeseries",
    header=None,
    engine="openpyxl",
)
print(f"D-Timeseries: {len(daily_timeseries)} rows, {daily_timeseries.shape[1]} columns")

selected_funds = match_daily_total_return(selected_funds, daily_timeseries)

_review = [
    column
    for column in (
        "Name",
        "openfigi_name",
        "edgar_name",
        "ISIN",
        "daily_tr_name",
        "daily_tr_observations",
        "daily_tr_name_source",
    )
    if column in selected_funds.columns
]

name_gap = selected_funds.loc[
    selected_funds["has_daily_total_return"] & ~selected_funds["daily_tr_name_match"],
    _review,
]
name_gap_export = _output_dir() / "v2-no-matching-D-TIMESERIES-NAME.xlsx"
name_gap.to_excel(name_gap_export, index=False)
print(f"Wrote {len(name_gap)} rows to {name_gap_export}")


D-Timeseries: 2877 rows, 3035 columns
595 of 606 funds have a daily total return series
584 of 595 series names match Name, OpenFIGI, or EDGAR
Wrote 11 rows to /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/v2-no-matching-D-TIMESERIES-NAME.xlsx


In [ ]:
def _output_dir() -> Path:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            destination = directory / "output-data"
            destination.mkdir(parents=True, exist_ok=True)
            return destination
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


funds_without_timeseries = selected_funds.loc[~selected_funds["has_daily_total_return"]].copy()
funds_without_timeseries_export = _output_dir() / "v2-no-daily-total-return.xlsx"
funds_without_timeseries.to_excel(funds_without_timeseries_export, index=False)
print(f"Wrote {len(funds_without_timeseries)} rows to {funds_without_timeseries_export}")
